In [ ]:
# Local runtime probe only. No remote notebook runtime.
import os
import platform
import subprocess
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
expected_python = (repo_root / ".venv" / "Scripts" / "python.exe").resolve()
actual_python = Path(sys.executable).resolve()
kernel_still_wrong = actual_python != expected_python or "COLAB_RELEASE_TAG" in os.environ

print("LOCAL_KERNEL_FIXED", "NO" if kernel_still_wrong else "YES")
print("REAL_INTERPRETER_PATH", sys.executable)
print("REAL_PLATFORM", platform.platform())
print("CWD", repo_root)
print("KERNEL_STILL_WRONG", "YES" if kernel_still_wrong else "NO")

try:
    smi = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,driver_version,memory.total,memory.used,temperature.gpu,utilization.gpu", "--format=csv,noheader"],
        text=True,
        capture_output=True,
        timeout=20,
        check=False,
    )
    print("NVIDIA_SMI_WORKS", "YES" if smi.returncode == 0 else "NO")
    if smi.stdout.strip():
        print("NVIDIA_SMI_GPU", smi.stdout.strip().splitlines()[0])
except Exception as exc:
    print("NVIDIA_SMI_WORKS", "NO")
    print("NVIDIA_SMI_ERROR", type(exc).__name__, str(exc))

try:
    import torch
    cuda_visible = torch.cuda.is_available()
    print("TORCH_VERSION", torch.__version__)
    print("TORCH_CUDA_BUILT", torch.version.cuda)
    print("TORCH_CUDA_VISIBLE", "YES" if cuda_visible else "NO")
    print("REAL_GPU_NAME", torch.cuda.get_device_name(0) if cuda_visible else "NO_GPU")
except Exception as exc:
    cuda_visible = False
    print("TORCH_CHECK_ERROR", type(exc).__name__, str(exc))

try:
    import numpy as np
    import xgboost as xgb
    print("XGB_VERSION", xgb.__version__)
    if cuda_visible:
        X = np.asarray([[0.0, 1.0], [1.0, 0.0], [2.0, 1.0], [3.0, 0.0]], dtype=np.float32)
        y = np.asarray([0.0, 1.0, 1.5, 2.0], dtype=np.float32)
        model = xgb.XGBRegressor(n_estimators=2, max_depth=1, tree_method="hist", device="cuda", objective="reg:squarederror", n_jobs=1)
        model.fit(X, y)
        print("XGB_GPU_USABLE", "YES")
    else:
        print("XGB_GPU_USABLE", "NO")
except Exception as exc:
    print("XGB_GPU_USABLE", "NO")
    print("XGBOOST_CHECK_ERROR", type(exc).__name__, str(exc))